# timesfm v3 champion — registry pipeline inference

this notebook performs inference only:

```text
raw test.csv → w&b registry champion pipeline → predictions → kaggle submission + w&b inference artifact
```

feature engineering, stored history, seasonal/raw/residual/xreg forecasting and blending are all inside the registered pipeline.

In [ ]:
%pip install -q "timesfm[torch]==2.0.2" "wandb==0.28.0" "cloudpickle>=3,<4" "kaggle>=1.7,<2"

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import hashlib
import json
import subprocess
import time
from pathlib import Path

import cloudpickle
import numpy as np
import pandas as pd
import timesfm
import torch
import wandb
from google.colab import drive, userdata

drive.mount("/content/drive")

In [ ]:
CONFIG = {
    "data_dir": "/content/drive/MyDrive/walmart_competition_data",
    "output_dir": "/content/drive/MyDrive/walmart_competition_inference/timesfm",
    "download_dir": "/content/artifacts/timesfm_registry_pipeline",
    "wandb_entity": "kende23-n-a",
    "wandb_project": "Walmart-Recruiting---Store-Sales-Forecasting",
    "registry_uri": "wandb-registry-model/Walmart_TimesFM_Raw_Pipeline:champion",
    "pipeline_filename": "walmart_timesfm_v3_raw_pipeline.pkl",
    "submission_filename": "timesfm_v3_champion_submission.csv",
    "submission_artifact_name": "timesfm-v3-champion-inference",
    "submit_to_kaggle": False,
    "kaggle_competition": "walmart-recruiting-store-sales-forecasting",
    "kaggle_message": "TimesFM v3 W&B Registry champion pipeline",
}
DATA_DIR = Path(CONFIG["data_dir"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
DOWNLOAD_DIR = Path(CONFIG["download_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(CONFIG)

## load only the raw kaggle test data

In [ ]:
def read_competition_csv(name: str) -> pd.DataFrame:
    csv_path = DATA_DIR / name
    zip_path = DATA_DIR / f"{name}.zip"
    if csv_path.exists():
        return pd.read_csv(csv_path)
    if zip_path.exists():
        return pd.read_csv(zip_path)
    raise FileNotFoundError(
        f"Missing {csv_path} and {zip_path}. Download the Kaggle competition data first."
    )


test_raw = read_competition_csv("test.csv")
test_raw["Date"] = pd.to_datetime(test_raw["Date"])
required_columns = {"Store", "Dept", "Date", "IsHoliday"}
missing_columns = required_columns.difference(test_raw.columns)
if missing_columns:
    raise ValueError(f"Raw test data is missing columns: {sorted(missing_columns)}")
print(
    {
        "raw_test_rows": len(test_raw),
        "date_min": str(test_raw["Date"].min().date()),
        "date_max": str(test_raw["Date"].max().date()),
        "series": int(test_raw[["Store", "Dept"]].drop_duplicates().shape[0]),
    }
)

## authenticate and download the registry champion

In [ ]:
def get_colab_secret(name: str):
    try:
        return userdata.get(name)
    except Exception:
        return None


wandb_key = get_colab_secret("WANDB_API_KEY")
if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
else:
    wandb.login()

hf_token = get_colab_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

run = wandb.init(
    entity=CONFIG["wandb_entity"],
    project=CONFIG["wandb_project"],
    group="timesfm-inference",
    job_type="timesfm_registry_pipeline_inference",
    name="timesfm_v3_champion_registry_inference",
    config=CONFIG,
)
pipeline_artifact = run.use_artifact(CONFIG["registry_uri"])
pipeline_dir = Path(pipeline_artifact.download(root=str(DOWNLOAD_DIR)))
pipeline_path = pipeline_dir / CONFIG["pipeline_filename"]
if not pipeline_path.exists():
    candidates = sorted(pipeline_dir.glob("*.pkl"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            {"expected": str(pipeline_path), "candidates": [str(p) for p in candidates]}
        )
    pipeline_path = candidates[0]

with pipeline_path.open("rb") as file:
    pipeline = cloudpickle.load(file)
metadata = pipeline.metadata()
if type(pipeline).__name__ != "TimesFMRawPipeline":
    raise TypeError(f"Unexpected registered object: {type(pipeline).__name__}")
print(
    {
        "registry_artifact": pipeline_artifact.name,
        "pipeline_path": str(pipeline_path),
        "pipeline_type": type(pipeline).__name__,
        "metadata": metadata,
    }
)

## run the registered pipeline on raw test rows

In [ ]:
started = time.time()
predictions = np.asarray(pipeline.predict(test_raw), dtype=np.float64)
inference_minutes = (time.time() - started) / 60
if predictions.shape != (len(test_raw),):
    raise ValueError(
        f"Pipeline returned shape {predictions.shape}; expected {(len(test_raw),)}"
    )
if not np.isfinite(predictions).all():
    raise ValueError("Pipeline returned non-finite predictions.")
prediction_hash = hashlib.sha256(predictions.tobytes()).hexdigest()
prediction_metrics = {
    "inference/rows": len(predictions),
    "inference/minutes": inference_minutes,
    "inference/prediction_min": float(predictions.min()),
    "inference/prediction_mean": float(predictions.mean()),
    "inference/prediction_max": float(predictions.max()),
    "inference/zero_predictions": int((predictions == 0).sum()),
}
run.log(
    {
        **prediction_metrics,
        "inference/prediction_distribution": wandb.Histogram(predictions),
    }
)
print({**prediction_metrics, "prediction_sha256": prediction_hash})

## create, validate and log the kaggle submission

In [ ]:
generated_ids = (
    test_raw["Store"].astype(str)
    + "_"
    + test_raw["Dept"].astype(str)
    + "_"
    + test_raw["Date"].dt.strftime("%Y-%m-%d")
)
submission = pd.DataFrame({"Id": generated_ids, "Weekly_Sales": predictions})
try:
    sample_submission = read_competition_csv("sampleSubmission.csv")
except FileNotFoundError:
    sample_submission = None
if sample_submission is not None:
    if len(sample_submission) != len(submission):
        raise ValueError("Sample submission and predictions have different row counts.")
    if not sample_submission["Id"].astype(str).equals(submission["Id"]):
        raise ValueError("Generated IDs do not match sampleSubmission.csv order.")
if submission["Id"].duplicated().any():
    raise ValueError("Submission contains duplicate IDs.")
submission_path = OUTPUT_DIR / CONFIG["submission_filename"]
submission.to_csv(submission_path, index=False)
submission_hash = hashlib.sha256(submission_path.read_bytes()).hexdigest()
manifest = {
    "registry_uri": CONFIG["registry_uri"],
    "registry_artifact": pipeline_artifact.name,
    "pipeline_type": type(pipeline).__name__,
    "pipeline_metadata": metadata,
    "rows": len(submission),
    "inference_minutes": inference_minutes,
    "prediction_sha256": prediction_hash,
    "submission_sha256": submission_hash,
    "submission_path": str(submission_path),
}
manifest_path = OUTPUT_DIR / "timesfm_v3_inference_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
display(submission.head())
print({"submission_path": str(submission_path), "rows": len(submission)})

In [ ]:
preview = submission.assign(
    Store=test_raw["Store"].to_numpy(),
    Dept=test_raw["Dept"].to_numpy(),
    Date=test_raw["Date"].dt.strftime("%Y-%m-%d").to_numpy(),
    IsHoliday=test_raw["IsHoliday"].to_numpy(),
).head(1000)
run.log({"inference/prediction_preview": wandb.Table(dataframe=preview)})
result_artifact = wandb.Artifact(
    CONFIG["submission_artifact_name"],
    type="prediction",
    description="Kaggle submission produced by the registered TimesFM v3 champion raw pipeline",
    metadata=manifest,
)
result_artifact.add_file(str(submission_path))
result_artifact.add_file(str(manifest_path))
run.log_artifact(result_artifact, aliases=["latest", "champion-pipeline"])
run.summary.update(
    {
        **prediction_metrics,
        "registry/uri": CONFIG["registry_uri"],
        "registry/resolved_artifact": pipeline_artifact.name,
        "prediction_sha256": prediction_hash,
        "submission_sha256": submission_hash,
        "submission_path": str(submission_path),
    }
)

## optional kaggle upload

keep `submit_to_kaggle=False` until the csv has been inspected. kaggle credentials must already be configured in the colab runtime.

In [ ]:
if CONFIG["submit_to_kaggle"]:
    command = [
        "kaggle",
        "competitions",
        "submit",
        "-c",
        CONFIG["kaggle_competition"],
        "-f",
        str(submission_path),
        "-m",
        CONFIG["kaggle_message"],
    ]
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            "Kaggle submission failed. Check authentication and competition-rule acceptance."
        )
    run.summary["kaggle/submitted"] = True
else:
    print("Kaggle upload skipped. The submission CSV and W&B artifact are ready.")
    run.summary["kaggle/submitted"] = False

run.finish()
print(
    {
        "inference_complete": True,
        "registry_uri": CONFIG["registry_uri"],
        "submission_path": str(submission_path),
        "prediction_sha256": prediction_hash,
    }
)